# Fraud Analysis
Credit application fraud detection pipeline.


In [ ]:
import os
import polars as pl
import psycopg2
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

from src.fraud import run_all, to_dataframe, decide_applications, REJECTION_THRESHOLD

load_dotenv()

DATABASE_URL = os.environ['DATABASE_URL']
conn = psycopg2.connect(DATABASE_URL)

In [ ]:
# Load data from Neon
applications = pl.read_database('SELECT * FROM credit_applications', connection=conn)
transactions = pl.read_database('SELECT * FROM transactions', connection=conn)

print(f'Applications: {len(applications)}')
print(f'Transactions: {len(transactions)}')

In [ ]:
# EDA: category distribution
cat_counts = transactions.group_by('category').agg(pl.len().alias('cnt')).sort('cnt', descending=True)
print(cat_counts)

# Country distribution
country_dist = transactions.group_by('location_country').agg(pl.len().alias('cnt')).sort('cnt', descending=True)
print(country_dist)

In [ ]:
# Run all fraud rules using extracted modules
fraud_flags = run_all(transactions, applications)
results = to_dataframe(fraud_flags)

print(f'Total flags: {len(results)}')
print(f'Triggered: {results.filter(pl.col("triggered") == True).height}')

In [ ]:
# Summary by application
summary = results.group_by('application_id').agg([
    pl.col('triggered').sum().alias('rules_triggered'),
    pl.col('score').mean().alias('avg_score'),
    pl.col('score').max().alias('max_score'),
]).sort('rules_triggered', descending=True)

decisions = decide_applications(fraud_flags)
print(f'\nRejection threshold: {REJECTION_THRESHOLD}+ rules = rejected')
print(f'\nDecision summary:')
print(decisions.group_by('decision').agg(pl.len().alias('count')))

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Rules triggered per applicant
axes[0].bar(
    summary['application_id'].cast(str).to_list(), 
    summary['rules_triggered'].to_list(), 
    color='coral'
)
axes[0].set_title('Rules Triggered per Applicant')
axes[0].set_xlabel('Applicant ID')
axes[0].set_ylabel('# Rules Triggered')

# 2. Avg score by rule
rule_scores = results.group_by('rule_name').agg(pl.col('score').mean().alias('avg_score')).sort('avg_score', descending=True)
axes[1].barh(rule_scores['rule_name'].to_list(), rule_scores['avg_score'].to_list(), color='steelblue')
axes[1].set_title('Avg Score by Rule')

# 3. Triggered vs clean by rule
rule_triggered = results.group_by('rule_name').agg(pl.col('triggered').sum().alias('triggered_count'))
axes[2].bar(rule_triggered['rule_name'].to_list(), rule_triggered['triggered_count'].to_list(), color='indianred')
axes[2].set_title('Triggered Count by Rule')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of scores by applicant and rule
pivot_data = results.pivot(on='rule_name', index='application_id', values='score').sort('application_id')
rule_cols = [c for c in pivot_data.columns if c != 'application_id']
heatmap_data = pivot_data.select(rule_cols).to_numpy()
app_ids = pivot_data['application_id'].to_list()

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    heatmap_data,
    xticklabels=rule_cols,
    yticklabels=[f'Applicant {i}' for i in app_ids],
    annot=True, fmt='.0f', cmap='YlOrRd', ax=ax
)
ax.set_title('Fraud Score Heatmap by Applicant and Rule')
plt.tight_layout()
plt.show()

In [ ]:
# Write results back to Neon
import psycopg2.extras

cur = conn.cursor()

# Clear old results
cur.execute('DELETE FROM fraud_results')

for flag in fraud_flags:
    cur.execute(
        'INSERT INTO fraud_results (application_id, rule_name, triggered, score, details) VALUES (%s, %s, %s, %s, %s)',
        (flag['application_id'], flag['rule_name'], flag['triggered'], flag['score'], flag['details'])
    )

conn.commit()
print(f'Wrote {len(fraud_flags)} results to fraud_results table.')

In [ ]:
# Update application statuses based on decisions
for row in decisions.iter_rows(named=True):
    cur.execute(
        'UPDATE credit_applications SET status = %s WHERE id = %s',
        (row['decision'], row['application_id'])
    )

conn.commit()
print(f'Updated {len(decisions)} applications.')

In [ ]:
conn.close()
print('done ✓')